In [ ]:
#!/usr/bin/env python3
"""
Figure: 3x2 boxplots (Longitude left, Latitude right) for precipitation efficiency (ε)
- MCS: cyan
- Non-MCS: orange
- Boxes: IQR (25–75)
- Median: horizontal line
- Mean: dot
- Whiskers: 5–95th percentiles

ADDED (quantitative output):
1) DOMAIN-WIDE pooled stats printed ONCE (lon/lat pooled are identical by construction)
2) BIN-AVERAGED stats (equal weight per lon-bin vs lat-bin) using bin medians (these can differ)
3) Per-bin stats printed (optional; can be verbose)
4) CSV tables saved:
   - per-bin stats for lon and lat for each variable
   - bin-median summaries for lon and lat for each variable
"""

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import FixedLocator, FixedFormatter
import math
import csv

# -----------------------------
# USER INPUT
# -----------------------------
files = {
    "PE": {
        "MCS": "/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/MCS_PRECEFF/Obs_cwp_PRECEFF_mcs_20160809-20160909_Asia_timeavg.nc",
        "NonMCS": "/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/MCS_PRECEFF/Obs_cwp_PRECEFF_non_mcs_20160809-20160909_Asia_timeavg.nc",
        "var": "PRECEFF_TIMEAVG",
        "ylabel": r"$\epsilon_{cwp}$ [hr$^{-1}$]"
    },
    "CIW": {
        "MCS": "/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/MCS_PRECEFF/Obs_ciw_PRECEFF_mcs_20160809-20160909_Asia_timeavg.nc",
        "NonMCS": "/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/MCS_PRECEFF/Obs_ciw_PRECEFF_non_mcs_20160809-20160909_Asia_timeavg.nc",
        "var": "PRECEFF_TIMEAVG",
        "ylabel": r"$\epsilon_{i}$ [hr$^{-1}$]"
    },
    "CLW": {
        "MCS": "/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/MCS_PRECEFF/Obs_clw_PRECEFF_mcs_20160809-20160909_Asia_timeavg.nc",
        "NonMCS": "/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/MCS_PRECEFF/Obs_clw_PRECEFF_non_mcs_20160809-20160909_Asia_timeavg.nc",
        "var": "PRECEFF_TIMEAVG",
        "ylabel": r"$\epsilon_{\ell}$ [hr$^{-1}$]"
    }
}

# Binning / plotting controls
bin_width = 4.0
whis = (5, 95)
box_width_factor = 0.30
dx = 0.9

legend_fontsize = 20
tick_fontsize = 20
ylabel_fontsize = 25
xlabel_fontsize = 25
panel_label_fontsize = 25

# Print control (set False if too verbose)
PRINT_PER_BIN = True

# Domain bounds
LON_MIN, LON_MAX = 55.0, 115.0
LAT_MIN, LAT_MAX = -5.0, 43.0

# Output
out_png = "/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/MCS_Paper_Plots/FigureS3_Boxplots_MCS_vs_Non_MCS_longitude_latitude.png"
out_pdf = "/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/MCS_Paper_Plots/FigureS3_Boxplots_MCS_vs_Non_MCS_longitude_latitude.pdf"
out_stats_dir = "/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/MCS_Paper_Plots/Boxplot_stats_tables/"
os.makedirs(out_stats_dir, exist_ok=True)

# -----------------------------
# HELPERS
# -----------------------------
def _scale_to_per_hour(values, units_str):
    """Return values scaled to hr^-1 and a flag indicating scaling."""
    units = (units_str or "").strip().lower()
    if ("s-1" in units) or (units in {"1/s", "s^-1", "sec^-1", "per second"}):
        return values * 3600.0, True
    if ("h-1" in units) or (units in {"1/h", "hr^-1", "hour^-1", "per hour"}):
        return values, False
    # Heuristic if units missing
    med = np.nanmedian(values) if values.size else np.nan
    if np.isfinite(med) and (1e-6 <= med <= 1e-2):
        return values * 3600.0, True
    return values, False

def load_field(path, var):
    """Open dataset, standardize coords, convert ε to hr^-1."""
    ds = xr.open_dataset(path)
    da = ds[var]

    # standardize coords
    if "lat" not in da.coords and "latitude" in da.coords:
        da = da.rename({"latitude": "lat"})
    if "lon" not in da.coords and "longitude" in da.coords:
        da = da.rename({"longitude": "lon"})

    vals = da.values
    vals_hr, _ = _scale_to_per_hour(vals, da.attrs.get("units", ""))
    da_hr = xr.DataArray(vals_hr, coords=da.coords, dims=da.dims, attrs=da.attrs)
    da_hr.attrs["units"] = "hr^-1"

    lat = da_hr["lat"].values
    lon = da_hr["lon"].values

    ds.close()
    return da_hr, lat, lon

def normalize_lon_inplace(da_mcs, da_non):
    """Wrap lon if needed to [0,360) so domain slicing is consistent."""
    lon = da_mcs["lon"].values
    if (lon.min() < 0) or (lon.max() > 180):
        lon = np.mod(lon, 360.0)
        da_mcs = da_mcs.assign_coords(lon=lon)
        da_non = da_non.assign_coords(lon=lon)
    return da_mcs, da_non, lon

def make_bins(start, stop, width):
    edges = np.arange(start, stop + width, width)
    centers = 0.5 * (edges[:-1] + edges[1:])
    return edges, centers

def collect_by_bins(da2d, coord_vals, edges, dim_name):
    """
    Collect values in each coordinate bin by indexing along dim_name (lat or lon),
    then flattening remaining dims.
    """
    out = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (coord_vals >= lo) & (coord_vals < hi)
        if not np.any(mask):
            out.append(np.array([]))
            continue
        idx = np.where(mask)[0]
        vals = da2d.isel({dim_name: idx}).values.ravel()
        vals = vals[np.isfinite(vals)]
        out.append(vals)
    return out

def style_boxplot(bp, face, edge, lw=2):
    for p in bp["boxes"]:
        p.set_facecolor(face)
        p.set_edgecolor(edge)
    for k in ["whiskers", "caps", "medians"]:
        for line in bp[k]:
            line.set_color(edge)
            line.set_linewidth(lw)

def summarize_bins(bins):
    """Return list of dicts (one per bin) with boxplot-relevant summary stats."""
    stats = []
    for v in bins:
        if len(v) == 0:
            stats.append(dict(N=0, mean=np.nan, median=np.nan, p25=np.nan, p75=np.nan, p05=np.nan, p95=np.nan))
        else:
            stats.append(dict(
                N=int(len(v)),
                mean=float(np.nanmean(v)),
                median=float(np.nanmedian(v)),
                p25=float(np.nanpercentile(v, 25)),
                p75=float(np.nanpercentile(v, 75)),
                p05=float(np.nanpercentile(v, 5)),
                p95=float(np.nanpercentile(v, 95)),
            ))
    return stats

def pooled_stats(bins):
    """Pool all bin values and compute overall stats (grid-cell weighted)."""
    pooled_list = [v for v in bins if len(v) > 0]
    if len(pooled_list) == 0:
        return dict(N=0, mean=np.nan, median=np.nan, p25=np.nan, p75=np.nan, p05=np.nan, p95=np.nan)
    pooled = np.concatenate(pooled_list)
    pooled = pooled[np.isfinite(pooled)]
    if pooled.size == 0:
        return dict(N=0, mean=np.nan, median=np.nan, p25=np.nan, p75=np.nan, p05=np.nan, p95=np.nan)
    return dict(
        N=int(pooled.size),
        mean=float(np.nanmean(pooled)),
        median=float(np.nanmedian(pooled)),
        p25=float(np.nanpercentile(pooled, 25)),
        p75=float(np.nanpercentile(pooled, 75)),
        p05=float(np.nanpercentile(pooled, 5)),
        p95=float(np.nanpercentile(pooled, 95)),
    )

def summarize_bin_medians(stats_list):
    """
    Bin-averaged summary (equal weight per bin) using bin medians.
    This can differ between lon and lat because the bin sets differ.
    """
    meds = np.array([s["median"] for s in stats_list if np.isfinite(s["median"]) and s["N"] > 0])
    if meds.size == 0:
        return dict(Nbins=0, mean=np.nan, median=np.nan, p25=np.nan, p75=np.nan, p05=np.nan, p95=np.nan)
    return dict(
        Nbins=int(meds.size),
        mean=float(np.nanmean(meds)),
        median=float(np.nanmedian(meds)),
        p25=float(np.nanpercentile(meds, 25)),
        p75=float(np.nanpercentile(meds, 75)),
        p05=float(np.nanpercentile(meds, 5)),
        p95=float(np.nanpercentile(meds, 95)),
    )

def safe_ratio(a, b):
    """Compute a/b safely."""
    if not np.isfinite(a) or not np.isfinite(b) or b == 0:
        return np.nan
    return a / b

def write_stats_csv(csv_path, centers, stats_mcs, stats_non, coord_name, var_label):
    """Write per-bin stats to CSV, including delta/ratio of medians."""
    header = [
        f"{coord_name}_center",
        "MCS_N", "MCS_mean", "MCS_median", "MCS_p25", "MCS_p75", "MCS_p05", "MCS_p95",
        "NonMCS_N", "NonMCS_mean", "NonMCS_median", "NonMCS_p25", "NonMCS_p75", "NonMCS_p05", "NonMCS_p95",
        "Delta_median(MCS-NonMCS)", "Ratio_median(MCS/NonMCS)"
    ]
    with open(csv_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow([f"# Per-bin stats for {var_label} along {coord_name}"])
        w.writerow(header)
        for c, sm, sn in zip(centers, stats_mcs, stats_non):
            delta = sm["median"] - sn["median"] if (sm["N"] > 0 and sn["N"] > 0) else np.nan
            ratio = safe_ratio(sm["median"], sn["median"]) if (sm["N"] > 0 and sn["N"] > 0) else np.nan
            w.writerow([
                float(c),
                sm["N"], sm["mean"], sm["median"], sm["p25"], sm["p75"], sm["p05"], sm["p95"],
                sn["N"], sn["mean"], sn["median"], sn["p25"], sn["p75"], sn["p05"], sn["p95"],
                delta, ratio
            ])

def write_binmedian_summary_csv(csv_path, mcs_lon, non_lon, mcs_lat, non_lat, var_label):
    """
    Write bin-median summaries (equal weight per bin) to CSV for lon and lat.
    """
    header = [
        "summary_type", "Nbins",
        "MCS_binmedian_median", "MCS_binmedian_p25", "MCS_binmedian_p75",
        "NonMCS_binmedian_median", "NonMCS_binmedian_p25", "NonMCS_binmedian_p75",
        "Delta_median(MCS-NonMCS)", "Ratio_median(MCS/NonMCS)"
    ]
    with open(csv_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow([f"# Bin-median (equal-bin-weight) summaries for {var_label}"])
        w.writerow(header)

        # longitude
        delta_lon = mcs_lon["median"] - non_lon["median"] if (mcs_lon["Nbins"] > 0 and non_lon["Nbins"] > 0) else np.nan
        ratio_lon = safe_ratio(mcs_lon["median"], non_lon["median"]) if (mcs_lon["Nbins"] > 0 and non_lon["Nbins"] > 0) else np.nan
        w.writerow([
            "longitude_binmedians", mcs_lon["Nbins"],
            mcs_lon["median"], mcs_lon["p25"], mcs_lon["p75"],
            non_lon["median"], non_lon["p25"], non_lon["p75"],
            delta_lon, ratio_lon
        ])

        # latitude
        delta_lat = mcs_lat["median"] - non_lat["median"] if (mcs_lat["Nbins"] > 0 and non_lat["Nbins"] > 0) else np.nan
        ratio_lat = safe_ratio(mcs_lat["median"], non_lat["median"]) if (mcs_lat["Nbins"] > 0 and non_lat["Nbins"] > 0) else np.nan
        w.writerow([
            "latitude_binmedians", mcs_lat["Nbins"],
            mcs_lat["median"], mcs_lat["p25"], mcs_lat["p75"],
            non_lat["median"], non_lat["p25"], non_lat["p75"],
            delta_lat, ratio_lat
        ])

# -----------------------------
# FIGURE 3x2
# -----------------------------
fig, axes = plt.subplots(3, 2, figsize=(16, 12), sharey="row")

mcs_patch = mpatches.Patch(facecolor="cyan", edgecolor="blue", label="MCS")
non_patch = mpatches.Patch(facecolor="orange", edgecolor="#D55E00", label="Non-MCS")

panel_labels = [r"(a)", r"(b)", r"(c)", r"(d)", r"(e)", r"(f)"]
panel_idx = 0

for row, (key, info) in enumerate(files.items()):
    da_mcs, lat, lon = load_field(info["MCS"], info["var"])
    da_non, _, _     = load_field(info["NonMCS"], info["var"])
    da_mcs, da_non, lon = normalize_lon_inplace(da_mcs, da_non)

    # -------------------------
    # LONGITUDE (left column)
    # -------------------------
    ax_lon = axes[row, 0]
    edges_lon, centers_lon = make_bins(LON_MIN, LON_MAX, bin_width)
    bins_mcs_lon = collect_by_bins(da_mcs, lon, edges_lon, dim_name="lon")
    bins_non_lon = collect_by_bins(da_non, lon, edges_lon, dim_name="lon")

    widths = box_width_factor * bin_width
    pos_mcs_lon = centers_lon - dx / 2
    pos_non_lon = centers_lon + dx / 2

    bp_mcs = ax_lon.boxplot(
        bins_mcs_lon, positions=pos_mcs_lon, widths=widths,
        whis=whis, patch_artist=True, showfliers=False
    )
    style_boxplot(bp_mcs, "cyan", "blue")
    ax_lon.scatter(
        pos_mcs_lon,
        [np.nanmean(v) if len(v) else np.nan for v in bins_mcs_lon],
        s=18, color="blue", zorder=3
    )

    bp_non = ax_lon.boxplot(
        bins_non_lon, positions=pos_non_lon, widths=widths,
        whis=whis, patch_artist=True, showfliers=False
    )
    style_boxplot(bp_non, "orange", "#D55E00")
    ax_lon.scatter(
        pos_non_lon,
        [np.nanmean(v) if len(v) else np.nan for v in bins_non_lon],
        s=18, color="#D55E00", zorder=3
    )

    ax_lon.set_xlim(LON_MIN, LON_MAX)
    tick_vals_lon = np.arange(LON_MIN, LON_MAX + 1e-6, 5.0)
    ax_lon.xaxis.set_major_locator(FixedLocator(tick_vals_lon))
    ax_lon.xaxis.set_major_formatter(FixedFormatter([f"{t:.0f}" for t in tick_vals_lon]))
    ax_lon.grid(True, axis="y", linestyle="--", alpha=0.4)
    ax_lon.set_ylabel(info["ylabel"], fontsize=ylabel_fontsize)
    if row == 2:
        ax_lon.set_xlabel("Longitude (°E)", fontsize=xlabel_fontsize)

    ax_lon.text(
        0.02, 0.96, panel_labels[panel_idx],
        transform=ax_lon.transAxes, ha="left", va="top",
        fontsize=panel_label_fontsize, fontweight="bold",
        bbox=dict(facecolor="white", alpha=0.7, edgecolor="none", pad=2)
    )
    panel_idx += 1

    # -------------------------
    # LATITUDE (right column)
    # -------------------------
    ax_lat = axes[row, 1]
    edges_lat, centers_lat = make_bins(LAT_MIN, LAT_MAX, bin_width)
    bins_mcs_lat = collect_by_bins(da_mcs, lat, edges_lat, dim_name="lat")
    bins_non_lat = collect_by_bins(da_non, lat, edges_lat, dim_name="lat")

    pos_mcs_lat = centers_lat - dx / 2
    pos_non_lat = centers_lat + dx / 2

    bp_mcs = ax_lat.boxplot(
        bins_mcs_lat, positions=pos_mcs_lat, widths=widths,
        whis=whis, patch_artist=True, showfliers=False
    )
    style_boxplot(bp_mcs, "cyan", "blue")
    ax_lat.scatter(
        pos_mcs_lat,
        [np.nanmean(v) if len(v) else np.nan for v in bins_mcs_lat],
        s=18, color="blue", zorder=3
    )

    bp_non = ax_lat.boxplot(
        bins_non_lat, positions=pos_non_lat, widths=widths,
        whis=whis, patch_artist=True, showfliers=False
    )
    style_boxplot(bp_non, "orange", "#D55E00")
    ax_lat.scatter(
        pos_non_lat,
        [np.nanmean(v) if len(v) else np.nan for v in bins_non_lat],
        s=18, color="#D55E00", zorder=3
    )

    ax_lat.set_xlim(LAT_MIN, LAT_MAX)
    tick_vals_lat = np.arange(LAT_MIN, LAT_MAX + 1e-6, 5.0)
    ax_lat.xaxis.set_major_locator(FixedLocator(tick_vals_lat))
    ax_lat.xaxis.set_major_formatter(FixedFormatter([f"{t:.0f}" for t in tick_vals_lat]))
    ax_lat.grid(True, axis="y", linestyle="--", alpha=0.4)
    if row == 2:
        ax_lat.set_xlabel("Latitude (°N)", fontsize=xlabel_fontsize)

    ax_lat.text(
        0.02, 0.96, panel_labels[panel_idx],
        transform=ax_lat.transAxes, ha="left", va="top",
        fontsize=panel_label_fontsize, fontweight="bold",
        bbox=dict(facecolor="white", alpha=0.7, edgecolor="none", pad=2)
    )
    panel_idx += 1

    # -------------------------
    # STATISTICS: per-bin, pooled, and bin-median summaries
    # -------------------------
    stats_mcs_lon = summarize_bins(bins_mcs_lon)
    stats_non_lon = summarize_bins(bins_non_lon)
    stats_mcs_lat = summarize_bins(bins_mcs_lat)
    stats_non_lat = summarize_bins(bins_non_lat)

    # True domain-wide pooled stats (grid-cell weighted) – identical regardless of binning
    pooled_mcs = pooled_stats(bins_mcs_lon)
    pooled_non = pooled_stats(bins_non_lon)

    # Bin-averaged summaries (equal weight per bin) – can differ between lon and lat
    bm_mcs_lon = summarize_bin_medians(stats_mcs_lon)
    bm_non_lon = summarize_bin_medians(stats_non_lon)
    bm_mcs_lat = summarize_bin_medians(stats_mcs_lat)
    bm_non_lat = summarize_bin_medians(stats_non_lat)

    var_label = info["ylabel"]

    print("\n" + "=" * 100)
    print(f"{key} :: {var_label}")
    print("-" * 100)

    print("DOMAIN-WIDE (all grid cells pooled; independent of lon/lat binning)")
    print(f"  MCS     : median={pooled_mcs['median']:.3f}, IQR={pooled_mcs['p25']:.3f}–{pooled_mcs['p75']:.3f}, N={pooled_mcs['N']}")
    print(f"  Non-MCS : median={pooled_non['median']:.3f}, IQR={pooled_non['p25']:.3f}–{pooled_non['p75']:.3f}, N={pooled_non['N']}")
    print(f"  Δmedian = {(pooled_mcs['median'] - pooled_non['median']):.3f}")
    print(f"  Ratio   = {safe_ratio(pooled_mcs['median'], pooled_non['median']):.3f}")

    print("\nBIN-AVERAGED (equal weight per longitude bin; using bin medians)")
    print(f"  MCS     : median={bm_mcs_lon['median']:.3f}, IQR={bm_mcs_lon['p25']:.3f}–{bm_mcs_lon['p75']:.3f}, Nbins={bm_mcs_lon['Nbins']}")
    print(f"  Non-MCS : median={bm_non_lon['median']:.3f}, IQR={bm_non_lon['p25']:.3f}–{bm_non_lon['p75']:.3f}, Nbins={bm_non_lon['Nbins']}")
    print(f"  Δmedian = {(bm_mcs_lon['median'] - bm_non_lon['median']):.3f}")
    print(f"  Ratio   = {safe_ratio(bm_mcs_lon['median'], bm_non_lon['median']):.3f}")

    print("\nBIN-AVERAGED (equal weight per latitude bin; using bin medians)")
    print(f"  MCS     : median={bm_mcs_lat['median']:.3f}, IQR={bm_mcs_lat['p25']:.3f}–{bm_mcs_lat['p75']:.3f}, Nbins={bm_mcs_lat['Nbins']}")
    print(f"  Non-MCS : median={bm_non_lat['median']:.3f}, IQR={bm_non_lat['p25']:.3f}–{bm_non_lat['p75']:.3f}, Nbins={bm_non_lat['Nbins']}")
    print(f"  Δmedian = {(bm_mcs_lat['median'] - bm_non_lat['median']):.3f}")
    print(f"  Ratio   = {safe_ratio(bm_mcs_lat['median'], bm_non_lat['median']):.3f}")

    if PRINT_PER_BIN:
        print("\nPer-bin LONGITUDE (centers): median/IQR and MCS–Non differences")
        for c, sm, sn in zip(centers_lon, stats_mcs_lon, stats_non_lon):
            if sm["N"] == 0 and sn["N"] == 0:
                continue
            delta = sm["median"] - sn["median"] if (sm["N"] > 0 and sn["N"] > 0) else np.nan
            ratio = safe_ratio(sm["median"], sn["median"]) if (sm["N"] > 0 and sn["N"] > 0) else np.nan
            print(
                f"  Lon {c:5.1f}°E | "
                f"MCS med={sm['median']:.3f} (IQR {sm['p25']:.3f}–{sm['p75']:.3f}, N={sm['N']}) | "
                f"Non med={sn['median']:.3f} (IQR {sn['p25']:.3f}–{sn['p75']:.3f}, N={sn['N']}) | "
                f"Δ={delta:.3f}, Ratio={ratio:.3f}"
            )

        print("\nPer-bin LATITUDE (centers): median/IQR and MCS–Non differences")
        for c, sm, sn in zip(centers_lat, stats_mcs_lat, stats_non_lat):
            if sm["N"] == 0 and sn["N"] == 0:
                continue
            delta = sm["median"] - sn["median"] if (sm["N"] > 0 and sn["N"] > 0) else np.nan
            ratio = safe_ratio(sm["median"], sn["median"]) if (sm["N"] > 0 and sn["N"] > 0) else np.nan
            print(
                f"  Lat {c:5.1f}°N | "
                f"MCS med={sm['median']:.3f} (IQR {sm['p25']:.3f}–{sm['p75']:.3f}, N={sm['N']}) | "
                f"Non med={sn['median']:.3f} (IQR {sn['p25']:.3f}–{sn['p75']:.3f}, N={sn['N']}) | "
                f"Δ={delta:.3f}, Ratio={ratio:.3f}"
            )

    # -------------------------
    # SAVE CSV TABLES
    # -------------------------
    csv_lon = os.path.join(out_stats_dir, f"stats_{key}_longitude.csv")
    csv_lat = os.path.join(out_stats_dir, f"stats_{key}_latitude.csv")
    write_stats_csv(csv_lon, centers_lon, stats_mcs_lon, stats_non_lon, "lon", var_label)
    write_stats_csv(csv_lat, centers_lat, stats_mcs_lat, stats_non_lat, "lat", var_label)

    csv_binmed = os.path.join(out_stats_dir, f"binmedian_summary_{key}.csv")
    write_binmedian_summary_csv(csv_binmed, bm_mcs_lon, bm_non_lon, bm_mcs_lat, bm_non_lat, var_label)

# -----------------------------
# Legend (once)
# -----------------------------
axes[0, 1].legend(
    handles=[mcs_patch, non_patch],
    loc="upper right", frameon=False, fontsize=legend_fontsize
)

# Tick font sizes
for r in range(3):
    for c in range(2):
        axes[r, c].tick_params(axis="x", labelsize=tick_fontsize)
        axes[r, c].tick_params(axis="y", labelsize=tick_fontsize)

# Rotate bottom-row tick labels
for c in range(axes.shape[1]):
    plt.setp(axes[-1, c].get_xticklabels(), rotation=35, ha="right")

# Same y-range per row (auto)
for r in range(3):
    top = max(axes[r, 0].get_ylim()[1], axes[r, 1].get_ylim()[1])
    top = math.ceil(top / 0.5) * 0.5
    axes[r, 0].set_ylim(0, top)
    axes[r, 1].set_ylim(0, top)

# Clean spines and ticks
for ax in axes.ravel():
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="out")
    ax.yaxis.set_ticks_position("left")
    ax.xaxis.set_ticks_position("bottom")

# Hide x tick labels for top two rows
for r in range(len(axes) - 1):
    for c in range(axes.shape[1]):
        axes[r, c].tick_params(labelbottom=False)

plt.tight_layout()

# Save figure
plt.savefig(out_png, dpi=150, bbox_inches="tight")
plt.savefig(out_pdf, dpi=50, bbox_inches="tight")

